# GWB iniettato su dati con scattering α = 4.4, analizzati con α = 4.4

Dati simulati in `Simulations_GWB_A140_g433_sv44.ipynb` (seed `jax.random.key(1)`, cartella `sim_data_noiseGWB_A140_g433_sv44/`): red noise + DM + scattering generato con **α = 4.4** + **GWB** con log10_A = −14.0 e γ = 13/3.

L'analisi usa 6 parametri, **senza** termine GWB, e modella lo scattering con **α = 4.4**, cioè l'indice corretto.

L'unica approssimazione del modello è quindi il GWB non modellato: l'effetto che si misura qui è solo il suo. Il confronto chiave è con `chain_sim_sv44_ok` (notebook 09): stessi processi di rumore, stesso seed, stesso modello di analisi, ma senza GWB. La differenza tra le due posterior è tutta e sola dovuta al GWB.

Il caso con α = 4.0 nell'analisi, cioè con entrambe le approssimazioni insieme, sarà un notebook successivo.

## Setup comune

Copia delle celle 0, 1, 3, 4, 6, 7, 9 di `Likelihood.ipynb`. `logL.params` serve come riferimento per l'ordine delle colonne.

In [ ]:
import numpy as np
import os, glob, json
import pickle

import sys

# this is the path to the discovery folder
sys.path.insert(0, "./")
import discovery_new as ds

import matplotlib.pyplot as plt
import scipy
import jax.numpy as jnp

from scipy.stats import uniform
import re
from PTMCMCSampler.PTMCMCSampler import PTSampler as ptmcmc
from scipy.stats import uniform
import jax

import time
import emcee

In [ ]:
# path to the folder with the data
path = './dati/DR2low_feathers/'
s_dsfiles = np.sort(os.listdir(path))
psr = [ds.pulsar.Pulsar.read_feather(path + f'{psrfile}') for psrfile in s_dsfiles][9] # we are only analysing pulsar #9

In [ ]:
# data
model = [psr.residuals]
# timing model
model += [ds.makegp_timing(psr, svd=True)]
# white noise parameters
model += [ds.makenoise_measurement(psr, psr.noisedict, tnequad = True, ecorr=True)]
# red noise paramters
model.append(ds.makegp_fourier(psr, ds.powerlaw, psr.noisedict[psr.name + '_red_components'], T=ds.getspan(psr), name='red_noise' ))
# DM variation parameters
model.append(ds.makegp_fourier(psr, ds.powerlaw, psr.noisedict[psr.name + '_dm_components'], T=ds.getspan(psr), name='dm_gp', fourierbasis=ds.make_dmfourierbasis(alpha = 2.0, tndm = False)))
# scattering noise parameters
model.append(ds.makegp_fourier(psr, ds.powerlaw, psr.noisedict[psr.name + '_chrom_components'], T=ds.getspan(psr), name='sv_gp', fourierbasis=ds.make_dmfourierbasis(alpha = 4.0, tndm = False)))

In [ ]:
logL = ds.PulsarLikelihood(model).logL

In [ ]:
# annoying function needed because the function logL want a dictionary as input, but our sampler wants a likelihood with an array as input
# so we have to switch from one to another :(
def params_dict_from_array(x):
    # ensure x is a plain numpy array before slicing
    x = np.asarray(x)
    return dict(zip(logL.params, x))

In [ ]:
##### Cell for the prior distributions ###################################################################################

# dictionary with minimum and maximum value of the prior distribution
my_priordict = {}
my_priordict["(.*_)?sv_gp_gamma.*"] = [0, 7]
my_priordict["(.*_)?sv_gp_alpha.*"] = [0, 10]
my_priordict["(.*_)?sv_gp_log10_A.*"] = [-20, -11]
my_priordict["(.*_)?sw_log10_A.*"] = [-10, 1]
my_priordict["(.*_)?sw_gamma.*"] = [-5, 6]
my_priordict["(.*_)?dm_gp_gamma.*"] = [0, 7]
my_priordict["(.*_)?dm_gp_log10_A.*"] = [-20, -11]
my_priordict["(.*_)?red_noise_gamma.*"] = [0, 7]
my_priordict["(.*_)?red_noise_log10_A.*"] = [-20, -11]
my_priordict["(.*_)?sw_n_earth.*"] = [0, 30]


def build_priors(param_names, priordict):
    """
    Build a {param_name: scipy dist} dict with uniform prior distributions in the range
    """
    priors = {}
    unmatched = []
    for name in param_names:
        matched = False
        for pattern, (lo, hi) in priordict.items():
            if re.fullmatch(pattern, name):
                priors[name] = uniform(loc=lo, scale=hi - lo) ## this is where the uniform distribution is defined
                matched = True
                break
        if not matched:
            unmatched.append(name)

    if unmatched:
        print(f"Warning: no prior found for {len(unmatched)} parameters, "
              f"falling back to standard normal:\n  {unmatched}")
    return priors

priors = build_priors(logL.params, my_priordict)
priors

def log_prior(x):
    """Compute log prior"""
    params = params_dict_from_array(x)
    
    logp = 0.0
    for name, value in params.items():
        if name in priors:
            logp += priors[name].logpdf(value)
        else:
            # Default: standard normal prior
            if isinstance(value, np.ndarray):
                logp += -0.5 * np.sum(value**2)
            else:
                logp += -0.5 * value**2
    
    return logp

def prior_draw():
    """Draw from prior and return as flat array"""
    x = []
    for name in logL.params:
        x.append(priors[name].rvs())
    return np.array(x)

x0 = prior_draw()
x0

In [ ]:
@jax.jit # compile the likelihood in jax to make it faster
def _jit_logL(params_array):
    params = dict(zip(logL.params, params_array))
    return logL(params)

_jit_logL(x0)

def log_likelihood(x): 
    """
    Likelihood function that is given to the samples, it uses the jitted (=faster) version of the likelihood
    """
    x = np.asarray(x, dtype=np.float64)
    result = _jit_logL(jnp.array(x))
    return float(np.array(result))  # always a plain Python float

## Catene di confronto (lette da disco, nessun sampling)

- `chain_sim_sv44_ok`: sim con α = 4.4 **senza GWB**, analizzata con α = 4.4 (notebook 09) — il riferimento
- `chain`: dati reali DR2low, analizzati con α = 4.0

In [ ]:
def load_chain(path):
    c = np.loadtxt(path)[:, :-4]
    return c[len(c)//4:]           # rimuovo il 25% di burn-in

chain_sim_sv44_ok = load_chain('./test_sim_sv44_alpha44/chain_1.txt')
chain             = load_chain('./test/chain_1.txt')
print(chain_sim_sv44_ok.shape, chain.shape)

## Dati simulati: scattering α = 4.4 + GWB

In [ ]:
sim_gwb_sv44_psr = ds.pulsar.Pulsar.read_feather('sim_data_noiseGWB_A140_g433_sv44/J1640+2224')
print('nome       :', sim_gwb_sv44_psr.name)
print('N_TOA      :', len(sim_gwb_sv44_psr.toas))
print('std residui:', np.std(sim_gwb_sv44_psr.residuals))

# controllo: deve essere diversa dalla simulazione senza GWB (stesso seed, stesso alpha)
sim_sv44_psr = ds.pulsar.Pulsar.read_feather('sim_data_noise_sv44/J1640+2224')
print('residui identici a quelli senza GWB?', np.allclose(sim_gwb_sv44_psr.residuals, sim_sv44_psr.residuals))
print('std della differenza (contributo del GWB):', np.std(sim_gwb_sv44_psr.residuals - sim_sv44_psr.residuals))

## Analisi con α = 4.4 e senza termine GWB

In [ ]:
model_gwb_sv44 = [sim_gwb_sv44_psr.residuals]
model_gwb_sv44 += [ds.makegp_timing(sim_gwb_sv44_psr, svd=True)]
model_gwb_sv44 += [ds.makenoise_measurement(sim_gwb_sv44_psr, sim_gwb_sv44_psr.noisedict, tnequad=True, ecorr=True)]
model_gwb_sv44.append(ds.makegp_fourier(sim_gwb_sv44_psr, ds.powerlaw, sim_gwb_sv44_psr.noisedict[sim_gwb_sv44_psr.name + '_red_components'],
                                        T=ds.getspan(sim_gwb_sv44_psr), name='red_noise'))
model_gwb_sv44.append(ds.makegp_fourier(sim_gwb_sv44_psr, ds.powerlaw, sim_gwb_sv44_psr.noisedict[sim_gwb_sv44_psr.name + '_dm_components'],
                                        T=ds.getspan(sim_gwb_sv44_psr), name='dm_gp',
                                        fourierbasis=ds.make_dmfourierbasis(alpha=2.0, tndm=False)))
model_gwb_sv44.append(ds.makegp_fourier(sim_gwb_sv44_psr, ds.powerlaw, sim_gwb_sv44_psr.noisedict[sim_gwb_sv44_psr.name + '_chrom_components'],
                                        T=ds.getspan(sim_gwb_sv44_psr), name='sv_gp',
                                        fourierbasis=ds.make_dmfourierbasis(alpha=4.4, tndm=False)))   # <-- 4.4, come nei dati

logL_gwb_sv44 = ds.PulsarLikelihood(model_gwb_sv44).logL
logL_gwb_sv44.params

In [ ]:
assert list(logL_gwb_sv44.params) == list(logL.params), (logL_gwb_sv44.params, logL.params)
print('OK: 6 parametri, identici a logL.params (nessun termine GWB, alpha fisso)')

In [ ]:
def params_dict_from_array_gwb_sv44(x):
    x = np.asarray(x)
    return dict(zip(logL_gwb_sv44.params, x))

priors_gwb_sv44 = build_priors(logL_gwb_sv44.params, my_priordict)

def log_prior_gwb_sv44(x):
    params = params_dict_from_array_gwb_sv44(x)
    logp = 0.0
    for name, value in params.items():
        if name in priors_gwb_sv44:
            logp += priors_gwb_sv44[name].logpdf(value)
        else:
            logp += -0.5 * np.sum(np.asarray(value)**2)
    return logp

def prior_draw_gwb_sv44():
    return np.array([priors_gwb_sv44[name].rvs() for name in logL_gwb_sv44.params])

x0_gwb_sv44 = prior_draw_gwb_sv44()

In [ ]:
@jax.jit
def _jit_logL_gwb_sv44(params_array):
    return logL_gwb_sv44(dict(zip(logL_gwb_sv44.params, params_array)))

def log_likelihood_gwb_sv44(x):
    x = np.asarray(x, dtype=np.float64)
    return float(np.array(_jit_logL_gwb_sv44(jnp.array(x))))

print('logL iniziale =', log_likelihood_gwb_sv44(x0_gwb_sv44))

In [ ]:
sampler_gwb_sv44 = ptmcmc(
    ndim=len(logL_gwb_sv44.params),
    logl=log_likelihood_gwb_sv44,
    logp=log_prior_gwb_sv44,
    cov=np.eye(len(logL_gwb_sv44.params)) * 0.001,
    outDir='./test_gwb_A140_g433_sv44/',
    resume=True
)

### Sampling

Eseguire **una sola volta**, circa 2 ore. α è fisso, quindi il costo per passo è quello solito dei run a 6 parametri.

In [ ]:
print("Starting sampling (sim GWB A=-14.0 gamma=13/3 su scattering alpha=4.4, analizzata con alpha=4.4 e SENZA GWB)...")
sampler_gwb_sv44.sample(x0_gwb_sv44, Niter=1000000, thin=10)

## Catena e risultati

In [ ]:
chain_gwb_sv44 = load_chain('./test_gwb_A140_g433_sv44/chain_1.txt')
chain_gwb_sv44.shape

In [ ]:
import emcee
tau = emcee.autocorr.integrated_time(chain_gwb_sv44, has_walkers=False, quiet=True)
print('tau =', np.round(tau, 1))
print('Effective length =', int(len(chain_gwb_sv44) / tau.max()))

In [ ]:
# quanto sposta il GWB ogni parametro, a parita' di tutto il resto
truths_dict = {
    'J1640+2224_dm_gp_gamma': 6.471070651479037,
    'J1640+2224_dm_gp_log10_A': -15.7227863246785,
    'J1640+2224_red_noise_gamma': 0.13235259553929019,
    'J1640+2224_red_noise_log10_A': -13.212902548624186,
    'J1640+2224_sv_gp_gamma': 1.3259815720503758,
    'J1640+2224_sv_gp_log10_A': -15.888300265911965,
}
truth_values = [truths_dict[p] for p in logL.params]
clean_labels = [name.replace('J1640+2224_', '') for name in logL.params]

print(f"{'parametro':20s} {'vero':>8s} | {'senza GWB (nb 09)':>22s} | {'con GWB':>22s} | scost/sigma")
for i, name in enumerate(logL.params):
    a16, a50, a84 = np.percentile(chain_sim_sv44_ok[:, i], [16, 50, 84])
    b16, b50, b84 = np.percentile(chain_gwb_sv44[:, i], [16, 50, 84])
    s = np.sqrt(((a84 - a16)/2)**2 + ((b84 - b16)/2)**2)
    print(f"{name.replace('J1640+2224_', ''):20s} {truths_dict[name]:8.2f} | {a50:7.2f} [{a16:6.2f},{a84:6.2f}] | {b50:7.2f} [{b16:6.2f},{b84:6.2f}] | {(b50-a50)/s:6.2f}")

### Recupero dei parametri di rumore

In [ ]:
import corner

truths_dict = {
    'J1640+2224_dm_gp_gamma': 6.471070651479037,
    'J1640+2224_dm_gp_log10_A': -15.7227863246785,
    'J1640+2224_red_noise_gamma': 0.13235259553929019,
    'J1640+2224_red_noise_log10_A': -13.212902548624186,
    'J1640+2224_sv_gp_gamma': 1.3259815720503758,
    'J1640+2224_sv_gp_log10_A': -15.888300265911965,
}
truth_values = [truths_dict[p] for p in logL.params]
clean_labels = [name.replace('J1640+2224_', '') for name in logL.params]

fig = corner.corner(
    chain_gwb_sv44,
    labels=clean_labels,
    truths=truth_values,
    truth_color='#222222',
    show_titles=True,
    title_fmt='.2f',
    label_kwargs={'fontsize': 14},
    title_kwargs={'fontsize': 11},
    hist_kwargs={'density': True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    bins=40,
    color='#6A4C93',
)
fig.set_size_inches(16, 16)
fig.suptitle('J1640+2224 — scattering α=4.4 + GWB (A=-14.0, γ=13/3), analizzata con α=4.4 e SENZA GWB\nlinee nere = valori iniettati dei processi di rumore', fontsize=14, y=1.00)
fig.savefig('corner_gwb_A140_g433_sv44.png', dpi=150, bbox_inches='tight')

### Effetto del solo GWB: senza GWB (notebook 09) vs con GWB

Stessi processi di rumore, stesso seed, stesso modello di analisi con α = 4.4. L'unica differenza è il GWB nei dati.

In [ ]:
import corner

truths_dict = {
    'J1640+2224_dm_gp_gamma': 6.471070651479037,
    'J1640+2224_dm_gp_log10_A': -15.7227863246785,
    'J1640+2224_red_noise_gamma': 0.13235259553929019,
    'J1640+2224_red_noise_log10_A': -13.212902548624186,
    'J1640+2224_sv_gp_gamma': 1.3259815720503758,
    'J1640+2224_sv_gp_log10_A': -15.888300265911965,
}
truth_values = [truths_dict[p] for p in logL.params]
clean_labels = [name.replace('J1640+2224_', '') for name in logL.params]

# sim senza GWB, analisi alpha=4.4 (notebook 09)
fig = corner.corner(
    chain_sim_sv44_ok,
    labels=clean_labels,
    truths=truth_values,
    truth_color='#222222',
    show_titles=True,
    title_fmt='.2f',
    label_kwargs={'fontsize': 14},
    title_kwargs={'fontsize': 11},
    hist_kwargs={'density': True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    bins=40,
    color='#457B9D',
)

# sim con GWB, analisi alpha=4.4
corner.corner(
    chain_gwb_sv44,
    labels=clean_labels,
    truths=truth_values,
    show_titles=True,
    title_fmt='.2f',
    label_kwargs={'fontsize': 14},
    title_kwargs={'fontsize': 11},
    hist_kwargs={'density': True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    bins=40,
    color='#6A4C93',
    fig=fig
)

fig.set_size_inches(16, 16)
fig.suptitle('J1640+2224 — α=4.4 in dati e analisi: senza GWB (blu) vs con GWB A=-14.0, γ=13/3 (viola)', fontsize=13, y=1.00)
fig.savefig('corner_sv44_senza_vs_con_gwb.png', dpi=150, bbox_inches='tight')

### Confronti opzionali

Eseguibili solo se le catene corrispondenti esistono: i dati reali con α = 4.4 (notebook 10) e la stessa iniezione di GWB su scattering α = 4.0 (notebook 11).

In [ ]:
import os, corner

truths_dict = {
    'J1640+2224_dm_gp_gamma': 6.471070651479037,
    'J1640+2224_dm_gp_log10_A': -15.7227863246785,
    'J1640+2224_red_noise_gamma': 0.13235259553929019,
    'J1640+2224_red_noise_log10_A': -13.212902548624186,
    'J1640+2224_sv_gp_gamma': 1.3259815720503758,
    'J1640+2224_sv_gp_log10_A': -15.888300265911965,
}
truth_values = [truths_dict[p] for p in logL.params]
clean_labels = [name.replace('J1640+2224_', '') for name in logL.params]

opzionali = [
    ('./test_real_sv44/chain_1.txt',         '#2E86AB', 'DR2low reali, analisi α=4.4 (nb 10)',      'corner_reali_sv44_vs_gwb_sv44.png'),
    ('./test_sim_gwb_A140_g433/chain_1.txt', '#A06CD5', 'GWB su scattering α=4.0, analisi α=4.0 (nb 11)', 'corner_gwb_g433_sv40_vs_sv44.png'),
]

for path, colore, etichetta, fname in opzionali:
    if not os.path.exists(path):
        print('salto (catena assente):', path)
        continue
    altra = load_chain(path)
    fig = corner.corner(
        altra,
        labels=clean_labels,
        truths=truth_values,
        truth_color='#222222',
    show_titles=True,
    title_fmt='.2f',
    label_kwargs={'fontsize': 14},
    title_kwargs={'fontsize': 11},
    hist_kwargs={'density': True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    bins=40,
        color=colore,
    )
    corner.corner(
        chain_gwb_sv44,
        labels=clean_labels,
        truths=truth_values,
    show_titles=True,
    title_fmt='.2f',
    label_kwargs={'fontsize': 14},
    title_kwargs={'fontsize': 11},
    hist_kwargs={'density': True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    bins=40,
        color='#6A4C93',
        fig=fig
    )
    fig.set_size_inches(16, 16)
    fig.suptitle(f'J1640+2224 — {etichetta} vs GWB su scattering α=4.4 (viola)', fontsize=13, y=1.00)
    fig.savefig(fname, dpi=150, bbox_inches='tight')
    print('salvato', fname)

## Riepilogo

_Da compilare dopo il run._

- Spostamento dei parametri di rumore dovuto al solo GWB: …
- Quale processo assorbe il GWB (atteso: il red noise): …
- Confronto con l'iniezione equivalente su scattering α = 4.0 (notebook 11): …
- Prossimo passo: stessi dati analizzati con α = 4.0, per vedere come si combinano le due approssimazioni.